In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

In [ ]:
import torch
from datasets import load_dataset
from stf_data_utils import *
from torch.utils.data import Dataset, DataLoader

In [1]:
from datasets import load_dataset
ds = load_dataset("vwxyzjn/summarize_from_feedback_tldr_3_filtered")

In [3]:
ds['train']

Dataset({
    features: ['id', 'subreddit', 'title', 'post', 'summary'],
    num_rows: 116722
})

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(
    "EleutherAI/pythia-1b"
)

tokenizer.add_special_tokens({
    "pad_token": "[PAD]"
})

# IMPORTANT: usually the exisitng works use pad from left but acc to the N+ paper implemenation we should pad from righ
tokenizer.padding_side = "right"


dataset = load_dataset(
    "vwxyzjn/summarize_from_feedback_tldr_3_filtered",
    # split="train"
)


train_dataset = TLDRDataset(
    dataset['train'],
    tokenizer
)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
)

test_dataset = TLDRDataset(
    dataset['test'],
    tokenizer
)
test_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
)


In [ ]:
dataset['train']

In [ ]:
batch = next(iter(train_loader))

print(batch["query_token"].shape)
# (B, 512)

print(batch["reference_response_token"].shape)
# (B, 53)

print(batch["query_reference_response_token"].shape)
# (B, 565)

In [ ]:
EPOCHS       = 1
LR           = 3e-6
WARMUP_STEPS = 200

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * EPOCHS,
)

device = 'cuda'
model.resize_token_embeddings(len(tokenizer))


In [ ]:
def sft_loss(model, batch):

    ids = batch["query_reference_response_token"] # this is the query + response token, the seuqnece is right padded

    # print("shape:", ids.shape)
    # print("min:", ids.min().item())
    # print("max:", ids.max().item())
    # print("pad_token_id:", tokenizer.pad_token_id)
    # print("tokenizer len:", len(tokenizer))
    # print("model vocab:", model.get_input_embeddings().num_embeddings)

    input_ids = ids.to(device)
    labels    = input_ids.clone()

    query_len = batch["query_token"].shape[1]          # 512 — mask the entire prompt
    labels[:, :query_len] = -100                       # mask query
    labels[input_ids == tokenizer.pad_token_id] = -100 # mask right-side padding

    return model(input_ids=input_ids, labels=labels).loss

In [ ]:
import torch
print(torch.cuda.is_available())
s = torch.tensor([1]).to("cuda")

In [ ]:
import wandb
wandb.init(
    project="gpt2_finetuning",
    name=f"gpt2",
    mode="online",
    # config={
    #     "stage": stage,
    #     "learning_rate": lr,
    #     "epochs": self.NUM_EPOCHS,
    #     "grad_accum": self.GRAD_ACCUM_STEPS,
    #     "dataset": "localized_narratives",
    # }
)

In [ ]:
# model = model.to(device)
# for epoch in range(EPOCHS):

#     # train
#     model.train()
#     total_loss = 0
#     for step, batch in tqdm(enumerate(train_loader), total = len(train_loader), desc=f"Epoch {epoch+1} train"):
#         # batch = batch.to(device)
#         loss = sft_loss(model, batch)
#         optimizer.zero_grad()
#         loss.backward()
#         nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()
#         scheduler.step()
#         total_loss += loss.item()

#         wandb.log(
#             {
#                 "loss": loss.item(),
#                 "step": step
#             }
#         )
#     # eval
#     model.eval()
#     eval_loss = 0
#     with torch.no_grad():
#         for batch in tqdm(test_loader, desc=f"Epoch {epoch+1} eval"):
#             batch = batch.to(device)
#             eval_loss += sft_loss(model, batch).item()

#     print(f"Epoch {epoch+1} | train_loss: {total_loss/len(train_loader):.4f} "
#           f"| eval_loss: {eval_loss/len(test_loader):.4f}")

#     model.save_pretrained(f"sft_epoch{epoch+1}")
#     tokenizer.save_pretrained(f"sft_epoch{epoch+1}")

In [ ]:
model = model.to(device)

gradient_accumulation_steps = 4

for epoch in range(EPOCHS):

    # train
    model.train()
    total_loss = 0

    optimizer.zero_grad()

    for step, batch in tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch+1} train"
    ):

        loss = sft_loss(model, batch)

        # normalize loss for accumulation
        loss = loss / gradient_accumulation_steps

        loss.backward()

        if (
            (step + 1) % gradient_accumulation_steps == 0
            or (step + 1) == len(train_loader)
        ):

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * gradient_accumulation_steps

        wandb.log(
            {
                "loss": loss.item() * gradient_accumulation_steps,
                "step": step
            }
        )

    # eval
    model.eval()
    eval_loss = 0

    # tr
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch+1} eval"):
            # batch = batch.to(device)
            eval_loss += sft_loss(model, batch).item()

    print(
        f"Epoch {epoch+1} | train_loss: {total_loss/len(train_loader):.4f} "
        f"| eval_loss: {eval_loss/len(test_loader):.4f}"
    )

    model.save_pretrained(f"sft_epoch{epoch+1}")
    tokenizer.save_pretrained(f"sft_epoch{epoch+1}")

In [ ]:
# load from a saved checkpoint
model = AutoModelForCausalLM.from_pretrained("sft_epoch1_dnd").to(device)
tokenizer = AutoTokenizer.from_pretrained("sft_epoch1_dnd")

# or load the base pretrained model
# model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
# tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
print(sum(p.numel() for p in model.parameters()))

In [ ]:

eval_loss = 0
model = model.to(device)
with torch.no_grad():
    for step, batch in tqdm(enumerate(test_loader), total=len(test_loader) ,desc=f"Epoch {1} eval"):
        # batch = batch.to(device)
        eval_loss = sft_loss(model, batch).item()
        # print(eval_loss)
        wandb.log(
            {
                "loss_eval": eval_loss,
                "step_eval": step
            }
        )
print(
    # f"Epoch {epoch+1} | train_loss: {total_loss/len(train_loader):.4f} "
    f"| eval_loss: {eval_loss/len(test_loader):.4f}"
)

In [ ]:
model.save_pretrained(f"sft_epoch{epoch+1}")
tokenizer.save_pretrained(f"sft_epoch{epoch+1}")